<a href="https://colab.research.google.com/github/Santibareiro27/Inteligencia-Computacional/blob/borges/RA1_LAB1/Datos_Laboratorio_N%C2%B01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiencia 1 — Sistema de Alerta Temprana para ACV en Atención Primaria
## Inteligencia Computacional — IC415 | RA1 Laboratorio N° 1

### Contexto del problema

Una red de centros de atención primaria de la salud en una provincia con alta dispersión geográfica enfrenta un problema crítico: los pacientes de zonas rurales suelen llegar a la guardia hospitalaria cuando un ACV (accidente cerebrovascular) ya ocurrió, con la ventana terapéutica cerrada. El tiempo es el factor determinante en el daño neurológico irreversible.

Se implementó un clasificador binario que, dado el perfil clínico y demográfico de un paciente durante una consulta de control, emite una alerta de derivación a especialista si el riesgo de ACV es elevado. **La alerta no reemplaza al médico** — actúa como señal de apoyo a la decisión clínica, especialmente en contextos rurales donde el acceso a especialistas es limitado.

**Dataset empleado:** ~5100 pacientes con 11 variables clínicas y demográficas. Variable objetivo: `apoplejia` (0 = sin ACV, 1 = ACV).

---

### Estructura del notebook

1. Análisis Exploratorio de Datos (EDA)
2. Pipeline de Preprocesamiento
3. Entrenamiento y Comparativa de Modelos
4. Evaluación del Modelo Seleccionado y Conclusiones


## Librerías y configuración global

In [1]:
# ─── Librerías ─── (importación única, sin redundancias)
import sys, os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn — preprocesamiento
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split, GridSearchCV

# Sklearn — modelos
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Sklearn — métricas
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve, roc_curve
)

# Imbalanced learning
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 110
print("Librerías importadas correctamente.")


Librerías importadas correctamente.


---
## 1. Análisis Exploratorio de Datos (EDA)

El análisis exploratorio se realizó sobre el dataset **crudo**, antes de cualquier transformación. El objetivo fue comprender la estructura de los datos, detectar anomalías y fundamentar las decisiones del pipeline de preprocesamiento.

> **Nota metodológica:** Toda codificación en este apartado se realizó exclusivamente para visualización. El pipeline de modelado (Sección 2) aplica su propio encoding sin data leakage.


### 1.1 Carga del dataset

In [ ]:
# Carga del dataset desde GitHub
# CORRECCIÓN: se usa la URL raw (no la URL de la página HTML del blob de GitHub)
url = "https://raw.githubusercontent.com/Santibareiro27/Inteligencia-Computacional/main/RA1_LAB1/datos_acv.csv"
df_acv = pd.read_csv(url)

print(f"Dimensiones del dataset: {df_acv.shape}")
print(f"\nColumnas y tipos de datos:")
print(df_acv.dtypes)
print(f"\nPrimeras filas:")
display(df_acv.head())


### 1.2 Variable objetivo — Análisis de desbalanceo de clases

Se analizó la distribución de la variable objetivo `apoplejia` para caracterizar el grado de desbalanceo del dataset. Esta etapa resultó fundamental para definir la estrategia de evaluación del modelo.

**Sobre el desbalanceo y la trampa del accuracy:**

El dataset presentó un severo desbalanceo: aproximadamente el 4.9% de los registros corresponden a pacientes con ACV. Este escenario tiene implicancias críticas para la evaluación.

Un modelo que predice siempre "sin ACV" para cualquier paciente obtiene:
- **Accuracy: ~95%** ✓ — parece excelente
- **Recall: 0%** ✗ — es completamente inútil clínicamente

Por este motivo, el accuracy fue descartado como métrica principal. El sistema no busca ser correcto en la mayoría silenciosa; busca detectar los casos de riesgo real. Ante el director médico, un accuracy del 95% sin análisis de recall no comunica nada útil sobre la capacidad del sistema para cumplir su propósito.

Las estrategias adoptadas para contrarrestar el desbalanceo fueron:
- Aplicación de **SMOTE** exclusivamente sobre el conjunto de entrenamiento para balancear las clases durante el aprendizaje.
- Optimización de **Recall** como métrica principal en la búsqueda de hiperparámetros.
- Análisis de umbral de decisión para maximizar la detección de casos positivos.
- Uso de la **matriz de confusión** y la **curva Precision-Recall** como métricas de evaluación principales.


In [ ]:
# Distribución de la variable objetivo: apoplejía
conteo_acv = df_acv['apoplejia'].value_counts()
prevalencia = df_acv['apoplejia'].mean()

print(f"Pacientes con ACV (apoplejía = 1): {conteo_acv[1]}  ({prevalencia*100:.2f}%)")
print(f"Pacientes sin ACV (apoplejía = 0): {conteo_acv[0]}  ({(1-prevalencia)*100:.2f}%)")
print(f"\nRatio de desbalanceo: 1:{conteo_acv[0]//conteo_acv[1]} (positivos:negativos)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Conteo absoluto
bars = axes[0].bar(['Sin ACV (0)', 'Con ACV (1)'], [conteo_acv[0], conteo_acv[1]],
                   color=['#4C9BE8', '#E85D5D'], width=0.5)
axes[0].set_title('Distribución de casos de ACV', fontweight='bold')
axes[0].set_ylabel('Cantidad de pacientes')
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 str(int(bar.get_height())), ha='center', fontweight='bold')

# Porcentaje
axes[1].pie([conteo_acv[0], conteo_acv[1]],
            labels=[f'Sin ACV\n({(1-prevalencia)*100:.1f}%)', f'Con ACV\n({prevalencia*100:.1f}%)'],
            colors=['#4C9BE8', '#E85D5D'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Proporción de clases', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n⚠ DESBALANCEO SEVERO: Solo el ~5% del dataset tiene ACV.")
print("→ El Accuracy no es una métrica válida en este contexto.")
print("→ Un modelo que siempre predice 'sin ACV' tendría ~95% de accuracy y 0% de recall.")


### 1.3 Análisis de valores faltantes

Se analizaron los valores faltantes del dataset y su relación con la variable objetivo. Este análisis orientó tanto la estrategia de imputación como la identificación de posibles sesgos introducidos por los datos ausentes.

Se identificó que `imc` presentó el mayor porcentaje de nulos (~3.9%). Al analizar la tasa de ACV entre los registros con IMC faltante, no se observaron diferencias significativas respecto a la tasa global, lo que sugirió que los nulos en esta variable son mayormente **aleatorios (MAR)**, validando el uso de imputación KNN.

La variable `estado_fumador` presentó nulos **implícitos** bajo la categoría `'desconocido'`. Este caso ameritó un análisis adicional que se desarrolla en la Sección 1.8, dado que la naturaleza de la ausencia del dato podría no ser aleatoria.


In [ ]:
# Análisis de nulos y su relación con la variable objetivo
faltantes = df_acv.isnull().sum()
pct_faltantes = (faltantes / len(df_acv) * 100).round(2)

conteo_nulos_con_acv = {}
pct_acv_en_nulos = {}
for col in df_acv.columns:
    nulos = df_acv[df_acv[col].isnull()]
    n = len(nulos)
    conteo_nulos_con_acv[col] = int(nulos['apoplejia'].sum()) if n > 0 else 0
    pct_acv_en_nulos[col] = round(nulos['apoplejia'].mean() * 100, 2) if n > 0 else 0.0

resumen_nulos = pd.DataFrame({
    'Nulos totales': faltantes,
    '% del dataset': pct_faltantes,
    'Nulos con ACV': pd.Series(conteo_nulos_con_acv),
    '% ACV entre nulos': pd.Series(pct_acv_en_nulos)
})

print("Resumen de valores faltantes y su relación con ACV:")
display(resumen_nulos[resumen_nulos['Nulos totales'] > 0])

print("\n→ El IMC es la variable con más nulos (~3.9%). Su distribución entre nulos")
print("  no difiere significativamente de la tasa global de ACV, lo que sugiere")
print("  que los nulos en IMC son mayormente aleatorios (MAR), haciendo válida la imputación.")
print("\n→ 'estado_fumador' tiene nulos implícitos (categoría 'desconocido'),")
print("  que serán tratados como NaN. Ver análisis ético en Sección 5 — Pregunta 7.")


### 1.4 Frecuencia de variables categóricas y relación con ACV

Se calculó la tasa de ACV por categoría de cada variable categórica para identificar patrones y evaluar la relevancia predictiva de cada una. Este análisis también permitió detectar variables con potencial sesgo ético en su uso como predictores médicos.

**Variables con dudas éticas o clínicas identificadas:**

- **`genero`:** Se observó correlación estadística entre género y tasa de ACV. Sin embargo, incluir el género como predictor implica que el sistema trataría diferente a hombres y mujeres con idéntico perfil clínico. Aunque el género tiene correlación con el riesgo cardiovascular, su inclusión en sistemas automáticos de salud puede generar inequidad sanitaria si un grupo resulta sistemáticamente sub-alertado. Se mantuvo en el modelo con observación activa; en una versión productiva debería auditarse si el modelo presenta disparidades por género en los falsos negativos.

- **`casado_alguna_vez`:** Mostró correlación baja con el target. Se evaluó que esta variable actúa como proxy de edad y red de apoyo social, sin ser un factor causal del ACV. Su inclusión en un modelo médico podría discriminar a pacientes por su historia personal. Se decidió su eliminación en la etapa de selección de features (Sección 1.10).


In [ ]:
cat_features = ['genero', 'hipertension', 'enfermedad_corazon', 'casado_alguna_vez',
                'tipo_trabajo', 'tipo_residencia', 'estado_fumador']

print("Tasa de ACV por categoría de cada feature:\n")
for col in cat_features:
    if col not in df_acv.columns:
        continue
    total = df_acv[col].value_counts()
    con_acv = df_acv[df_acv['apoplejia'] == 1].groupby(col).size()
    tasa = (con_acv / total * 100).round(2)
    resumen = pd.DataFrame({
        'Total pacientes': total,
        'Con ACV': con_acv,
        'Tasa ACV (%)': tasa
    }).sort_values('Tasa ACV (%)', ascending=False)
    print(f"── {col.upper()} ──")
    print(resumen.to_string())
    print()


### 1.5 Visualización de distribuciones — justificación de eliminación de categorías

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Género vs ACV
sns.countplot(data=df_acv, x='genero', hue='apoplejia',
              palette={0: '#4C9BE8', 1: '#E85D5D'}, ax=axes[0])
axes[0].set_title('Distribución por Género vs ACV', fontweight='bold')
axes[0].set_xlabel('Género')
axes[0].set_ylabel('Cantidad de pacientes')
axes[0].legend(title='ACV', labels=['Sin ACV', 'Con ACV'])

# Tipo trabajo vs ACV
sns.countplot(data=df_acv, x='tipo_trabajo', hue='apoplejia',
              palette={0: '#4C9BE8', 1: '#E85D5D'}, ax=axes[1])
axes[1].set_title('Distribución por Tipo de Trabajo vs ACV', fontweight='bold')
axes[1].set_xlabel('Tipo de Trabajo')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='ACV', labels=['Sin ACV', 'Con ACV'])

plt.tight_layout()
plt.show()

# Reporte de categorías a eliminar
n_otro = len(df_acv[df_acv['genero'] == 'otro'])
n_nunca_trabajo = len(df_acv[df_acv['tipo_trabajo'] == 'nunca_trabajo'])
print(f"Registros con género 'otro': {n_otro}")
print(f"Registros con tipo_trabajo 'nunca_trabajo': {n_nunca_trabajo}")


### 1.6 Eliminación de categorías — justificación

Se eliminaron dos categorías con representación estadísticamente insuficiente:

**`genero = 'otro'`:** Con un único registro en el dataset, el modelo no disponía de evidencia suficiente para aprender ningún patrón asociado a esta categoría. Su inclusión introduciría ruido sin valor predictivo.

> **Nota ética:** Esta decisión se reconoce como una limitación. En un dataset representativo con mayor cantidad de individuos de género no binario, esta categoría debería incluirse explícitamente. En producción, se recomienda recolectar más datos antes de excluir grupos demográficos del sistema.

**`tipo_trabajo = 'nunca_trabajo'`:** Categoría de muy baja representación. Los subgrupos que la componen (niños, jubilados, personas con discapacidad) tienen perfiles de riesgo muy distintos entre sí. Incluirlos como una sola categoría hubiera introducido ruido en el modelo sin aportar valor predictivo coherente.


In [ ]:
n_antes = len(df_acv)
df_acv = df_acv[df_acv['genero'] != 'otro'].copy()
df_acv = df_acv[df_acv['tipo_trabajo'] != 'nunca_trabajo'].copy()
df_acv = df_acv.reset_index(drop=True)
n_despues = len(df_acv)

print(f"Registros eliminados: {n_antes - n_despues}")
print(f"Dimensiones resultantes: {df_acv.shape}")


### 1.7 Análisis de variables continuas — detección de outliers

Se analizó la distribución de las variables continuas `edad`, `nivel_glucosa` e `imc` tanto globalmente como segmentadas por la variable objetivo. El objetivo fue identificar valores atípicos que requirieran tratamiento previo al entrenamiento.

Los boxplots mostraron la presencia de valores extremos en `imc` y `nivel_glucosa`. Se graficaron histogramas focalizados en los pacientes con ACV para determinar umbrales de corte clínicamente justificables.


In [ ]:
# Verificación de edades inválidas
edades_invalidas = df_acv[(df_acv['edad'] < 0) | (df_acv['edad'] > 120)]
print(f"Registros con edad inválida (< 0 ó > 120): {len(edades_invalidas)}")
if len(edades_invalidas) == 0:
    print("→ Sin edades fuera del rango válido.\n")

# Boxplots por variable continua, separados por presencia de ACV
continuas = ['edad', 'nivel_glucosa', 'imc']
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('Variables Continuas — Distribución global y por presencia de ACV', fontsize=13, fontweight='bold')

for i, col in enumerate(continuas):
    sns.boxplot(data=df_acv, y=col, ax=axes[0, i], color='#87CEEB')
    axes[0, i].set_title(f'{col} — distribución global')

    sns.boxplot(data=df_acv, x='apoplejia', y=col, ax=axes[1, i],
                palette={0: '#4C9BE8', 1: '#E85D5D'})
    axes[1, i].set_title(f'{col} — por ACV')
    axes[1, i].set_xticklabels(['Sin ACV', 'Con ACV'])

plt.tight_layout()
plt.show()


In [ ]:
# Análisis específico del IMC en pacientes con ACV — justificación del umbral
df_stroke = df_acv[df_acv['apoplejia'] == 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_stroke['imc'].dropna(), bins=60, kde=True, color='salmon', ax=axes[0])
axes[0].axvline(60, color='red', linestyle='--', label='Umbral IMC = 60')
axes[0].axvline(12, color='orange', linestyle='--', label='Umbral IMC = 12')
axes[0].set_title('IMC en pacientes con ACV', fontweight='bold')
axes[0].set_xlabel('IMC')
axes[0].legend()

sns.histplot(df_acv['nivel_glucosa'].dropna(), bins=60, kde=True, color='steelblue', ax=axes[1])
axes[1].axvline(40, color='red', linestyle='--', label='Umbral glucosa = 40 mg/dL')
axes[1].set_title('Nivel de glucosa (todos los pacientes)', fontweight='bold')
axes[1].set_xlabel('Nivel de glucosa (mg/dL)')
axes[1].legend()

plt.tight_layout()
plt.show()

n_imc_alto = len(df_stroke[df_stroke['imc'] >= 60])
print(f"Pacientes con ACV e IMC ≥ 60: {n_imc_alto} ({n_imc_alto/len(df_stroke)*100:.1f}%)")
print("→ IMC ≥ 60 ó < 12 se considera outlier clínico (fuera del rango de IMC humano realista)")
print("→ Glucosa < 40 mg/dL implica hipoglucemia severa, inconsistente con registro ambulatorio")


### 1.8 Tratamiento de outliers e implicancias éticas de los datos faltantes

**Criterios de outlier aplicados:**
- **IMC:** valores fuera del rango [12, 60] se consideraron fisiológicamente inconsistentes para un registro ambulatorio. Se convirtieron a `NaN` para imputación posterior, preservando el resto de la información del paciente.
- **Glucosa:** valores por debajo de 40 mg/dL indicarían hipoglucemia severa, incompatible con un registro de consulta rutinaria. Se convirtieron a `NaN`.
- **`estado_fumador = 'desconocido'`:** Se trató como dato faltante (`NaN`), no como una categoría válida de tabaquismo.

**Implicancias éticas de la estrategia de datos faltantes:**

La imputación KNN asume que el dato faltante es **aleatorio (MAR)**: la probabilidad de que el dato falte no depende del valor que tendría. Para `imc` esta suposición resultó razonable. Sin embargo, para `estado_fumador = 'desconocido'` esta suposición es cuestionable.

Es probable que "desconocido" sea un caso de **dato faltante no aleatorio (MNAR)**: los pacientes sin registro de tabaquismo pueden ser personas de zonas rurales con menor acceso al sistema de salud, adultos mayores con historiales incompletos, o pacientes con menor frecuencia de consultas preventivas. Todos estos subgrupos tienen, a priori, mayor riesgo de ACV.

Si la imputación les asigna el valor más frecuente ("nunca fumó"), el modelo subestimaría su riesgo, generando más falsos negativos en exactamente el grupo más vulnerable del estudio.

**Alternativas evaluadas:**
1. Mantener `'desconocido'` como categoría propia (no imputar), preservando la información de que el dato estaba ausente.
2. Agregar un feature binario `fumador_desconocido` (0/1) que permita al modelo aprender que la ausencia del dato es en sí misma informativa.
3. En producción, capacitar a los centros para registrar el estado fumador en todas las consultas.

Se optó por la imputación KNN como solución funcional para este prototipo, reconociendo explícitamente esta limitación. En una versión productiva, se recomendaría implementar la alternativa 2.


In [ ]:
# Los valores extremos se convierten en NaN para imputación posterior.
# Esto preserva el resto de la información del registro del paciente.

# IMC: rango clínico razonable [12, 60]
n_imc_out = ((df_acv['imc'] < 12) | (df_acv['imc'] > 60)).sum()
df_acv.loc[(df_acv['imc'] < 12) | (df_acv['imc'] > 60), 'imc'] = np.nan

# Glucosa: < 40 mg/dL es fisiológicamente inconsistente
n_gluc_out = (df_acv['nivel_glucosa'] < 40).sum()
df_acv.loc[df_acv['nivel_glucosa'] < 40, 'nivel_glucosa'] = np.nan

# 'desconocido' en estado_fumador = dato faltante, no categoría válida
# IMPLICANCIA ÉTICA: ver Sección 5 — Pregunta 7
n_desc = (df_acv['estado_fumador'] == 'desconocido').sum()
df_acv.loc[df_acv['estado_fumador'] == 'desconocido', 'estado_fumador'] = np.nan

print("Valores convertidos a NaN:")
print(f"  IMC outliers: {n_imc_out}")
print(f"  Glucosa outliers: {n_gluc_out}")
print(f"  Estado fumador 'desconocido': {n_desc}")
print("\nResumen de nulos tras tratamiento:")
print(df_acv[['imc', 'nivel_glucosa', 'estado_fumador']].isnull().sum())


### 1.9 Análisis de correlación — orientación para la selección de features

Se construyó una matriz de correlación de Spearman sobre una versión temporalmente codificada del dataset (solo para visualización, sin impacto en el pipeline). La correlación de Spearman es no paramétrica y más robusta frente a la distribución no normal de las variables.

Los resultados confirmaron que `edad`, `hipertensión`, `nivel_glucosa` e `imc` presentaron la mayor asociación estadística con la variable objetivo, alineándose con el conocimiento clínico establecido sobre factores de riesgo cardiovascular.


In [ ]:
# ─── CORRELACIÓN SOLO PARA EDA ───────────────────────────────────────────────
# Se crea una copia temporal codificada del dataset ÚNICAMENTE para visualizar
# correlaciones. Esta codificación NO se usa en el pipeline de modelado.
# El encoding del modelo se realiza en la Sección 2, solo sobre X_train.
# ─────────────────────────────────────────────────────────────────────────────

df_corr = df_acv.copy()

# Imputación simple temporal (mediana/moda) para poder calcular correlaciones
df_corr['imc'] = df_corr['imc'].fillna(df_corr['imc'].median())
df_corr['nivel_glucosa'] = df_corr['nivel_glucosa'].fillna(df_corr['nivel_glucosa'].median())
df_corr['estado_fumador'] = df_corr['estado_fumador'].fillna('nunca')

# Encoding temporal para visualización
df_corr['genero'] = df_corr['genero'].map({'hombre': 0, 'mujer': 1})
df_corr['estado_fumador'] = df_corr['estado_fumador'].map({'nunca': 0, 'fumaba': 1, 'fuma': 2})
df_corr['tipo_trabajo_enc'] = df_corr.groupby('tipo_trabajo')['apoplejia'].transform('mean')
df_corr['tipo_residencia_enc'] = df_corr['tipo_residencia'].map({'Rural': 0, 'Urbana': 1,
                                                                  'rural': 0, 'urbano': 1,
                                                                  'Rural ': 0, 'Urbana ': 1})
df_corr = df_corr.drop(columns=['id', 'tipo_trabajo', 'tipo_residencia'], errors='ignore')

corr_matrix = df_corr.corr(method='spearman')
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap completo
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=axes[0], cbar_kws={'shrink': 0.8})
axes[0].set_title('Matriz de Correlación de Spearman (EDA)', fontweight='bold')

# Correlación solo con la variable objetivo
corr_con_target = corr_matrix['apoplejia'].drop('apoplejia').sort_values()
colores_corr = ['#E85D5D' if c > 0 else '#4C9BE8' for c in corr_con_target]
corr_con_target.plot(kind='barh', ax=axes[1], color=colores_corr)
axes[1].set_title("Correlación de Spearman con 'apoplejia'", fontweight='bold')
axes[1].axvline(0.05, color='gray', linestyle='--', alpha=0.7, label='Umbral 0.05')
axes[1].axvline(-0.05, color='gray', linestyle='--', alpha=0.7)
axes[1].legend()
axes[1].set_xlabel('Correlación de Spearman')

plt.tight_layout()
plt.show()

print("\nCorrelación con la variable objetivo (orden descendente):")
print(corr_con_target.sort_values(ascending=False).to_string())


### 1.10 Selección de features — justificación técnica, clínica y operativa

Basándose en la matriz de correlación, criterios clínicos y la restricción operativa del entorno de atención primaria, se definió el conjunto final de features del modelo.

**Variables eliminadas:**

| Feature eliminado | Motivo |
|---|---|
| `id` | Identificador sin valor predictivo |
| `casado_alguna_vez` | Correlación < 0.10 con el target. Actúa como proxy de edad sin valor causal clínico. Dudas éticas sobre su uso como predictor médico (ver Sección 1.4). |
| `tipo_trabajo` | Correlación baja al controlar por edad. Categoría `nunca_trabajo` ya eliminada, reduciendo su utilidad. |
| `tipo_residencia` | Correlación muy baja con el target. Información capturada indirectamente por otros features. |

**Disponibilidad de variables en una consulta de 8-12 minutos:**

Se analizó qué variables del dataset pueden relevarse en el tiempo disponible de una consulta de control periódico. Esta restricción operativa es relevante para la viabilidad práctica del sistema en atención primaria.

| Variable | Disponible en consulta | Observación |
|---|---|---|
| `edad` | ✅ Inmediata | Siempre disponible |
| `genero` | ✅ Inmediata | Siempre disponible |
| `hipertensión` | ✅ Historia clínica | Disponible en consulta periódica |
| `enfermedad_corazón` | ✅ Historia clínica | Disponible en consulta periódica |
| `imc` | ✅ Medición en consultorio | Peso + talla: ~2 minutos |
| `estado_fumador` | ✅ Pregunta directa | < 1 minuto |
| **`nivel_glucosa`** | ⚠ **No inmediata** | Requiere análisis de sangre |

`nivel_glucosa` fue la única variable que requirió un estudio adicional para su disponibilidad en consulta. Sin embargo, muchos centros de atención primaria realizan glucemia capilar de rutina, lo que la haría disponible en la misma visita. Se mantuvo en el modelo por su alta correlación con el target y su disponibilidad en el dataset histórico.

> **Recomendación para producción:** entrenar y comparar dos versiones del modelo — una sin glucosa (para cuando no hay análisis disponible) y otra con glucosa (cuando el dato está disponible) — cuantificando el valor clínico de incluir la glucemia como parte del protocolo de screening.


In [ ]:
columnas_eliminar = ['id', 'casado_alguna_vez', 'tipo_trabajo', 'tipo_residencia']
df_acv = df_acv.drop(columns=[c for c in columnas_eliminar if c in df_acv.columns])

print(f"Features finales del modelo: {df_acv.drop('apoplejia', axis=1).columns.tolist()}")
print(f"Dimensiones: {df_acv.shape}")
display(df_acv.describe())


---
## 2. Pipeline de Preprocesamiento

> **Orden correcto (sin data leakage):**
> `SPLIT → ENCODE → IMPUTE (fit en train) → SCALE (fit en train) → SMOTE (solo en train)`
>
> **Error corregido respecto a la versión original:** el KNN Imputer se ajustaba sobre el dataset completo antes del split, filtrando información del conjunto de test hacia el entrenamiento e inflando artificialmente las métricas de evaluación.

---

### Definición de la estrategia de evaluación: tipos de error y métricas priorizadas

Antes de construir el pipeline, se definieron explícitamente las prioridades de evaluación en función del contexto clínico.

**¿Qué significa que el modelo "falle"?**

En este sistema existen dos tipos de error con consecuencias muy distintas:

- **Falso Negativo (FN) — caso no detectado:** el sistema no emite alerta para un paciente que sí tiene alto riesgo de ACV. En el contexto rural, esta persona puede no llegar al hospital a tiempo, con consecuencias neurológicas permanentes o fatales. **Este es el error más grave e irrecuperable.**

- **Falso Positivo (FP) — falsa alarma:** el sistema alerta sobre un paciente sin riesgo real. Genera una derivación innecesaria que consume tiempo y recursos médicos, pero no produce daño irreversible. El médico puede filtrar estas alertas con su criterio clínico.

**Métricas priorizadas:**

Por esta asimetría de costos, la métrica principal de optimización fue el **Recall (Sensibilidad)**:

> Recall = TP / (TP + FN) → *¿De todos los ACV reales, cuántos detectó el modelo?*

El GridSearchCV se configuró con `scoring='recall'`. Se descartó el accuracy por el severo desbalanceo de clases (~5% de positivos). Se descartó el F1-Score como métrica de optimización porque penaliza simétricamente los FP y FN, lo que no refleja la asimetría real del riesgo en este contexto clínico. El AUC-ROC se utilizó como métrica complementaria para evaluar la capacidad discriminatoria general del modelo.

> **Corrección respecto a la versión original:** se usaba `scoring='f1'`, que penaliza por igual ambos tipos de error. En el contexto rural con vidas en juego, esa penalización simétrica no es apropiada.


### 2.1 División train/test — PRIMER PASO

In [ ]:
# La variable objetivo se convierte a entero ANTES del split
# (garantiza que no haya valores float en y_train / y_test)
df_acv['apoplejia'] = df_acv['apoplejia'].astype(int)

X = df_acv.drop('apoplejia', axis=1)
y = df_acv['apoplejia']

# División estratificada: preserva la proporción de clases en train y test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f"Train: {X_train.shape[0]} registros | Positivos: {y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"Test:  {X_test.shape[0]} registros | Positivos: {y_test.sum()} ({y_test.mean()*100:.1f}%)")
print("\n→ La estratificación garantiza que el desbalanceo (~5% positivos)")
print("  se preserve tanto en train como en test.")


### 2.2 Encoding de variables categóricas

Se aplicó encoding sobre las variables categóricas restantes. Los mapeos utilizados son deterministas y conocidos a priori, por lo que no requirieron ajuste sobre los datos, evitando cualquier riesgo de data leakage.

Para `estado_fumador` se eligió encoding ordinal: `'nunca'=0`, `'fumaba'=1`, `'fuma'=2`, reflejando la acumulación de exposición al tabaco. Se reconoce como limitación que los ex-fumadores (`'fumaba'`) tienen un riesgo cardiovascular elevado que puede equipararse al de fumadores activos; en una versión mejorada, se evaluaría One-Hot Encoding para no imponer linealidad en esta relación.


In [ ]:
# Los mapeos son deterministas y conocidos a priori — no requieren fit sobre datos.
# Se aplica el mismo mapeo a train y test sin riesgo de leakage.

SMOKING_MAP = {
    'nunca': 0,
    'fumaba': 1,
    'fuma': 2
}
# Justificación del encoding ordinal:
# 'nunca' < 'fumaba' < 'fuma' refleja acumulación de exposición acumulada al tabaco.
# Limitación: ex-fumadores (fumaba) tienen riesgo cardiovascular elevado que puede
# equipararse al de fumadores activos. Esta decisión introduce una simplificación clínica.
# Alternativa más robusta: One-Hot Encoding si el modelo no asume linealidad.

GENDER_MAP = {'hombre': 0, 'mujer': 1}

def encode_categoricals(df):
    df = df.copy()
    df['genero'] = df['genero'].map(GENDER_MAP)
    df['estado_fumador'] = df['estado_fumador'].map(SMOKING_MAP)
    return df

X_train = encode_categoricals(X_train)
X_test = encode_categoricals(X_test)

print("Encoding aplicado. Tipos de datos tras encoding:")
print(X_train.dtypes)
print(f"\nNulos en X_train: {X_train.isnull().sum().sum()}")
print(f"Nulos en X_test:  {X_test.isnull().sum().sum()}")


### 2.3 Imputación KNN — ajustado exclusivamente en train

Se corrigió el error crítico de la versión original, donde el KNN Imputer se ajustaba sobre el dataset completo antes del split. La corrección aseguró que el imputer se ajuste **exclusivamente sobre `X_train`**, y que `X_test` solo reciba `transform()`, usando los parámetros aprendidos del entrenamiento.

El escalado temporal previo a la imputación garantizó que el KNN utilice distancias euclidianas equitativas entre variables de distintas escalas.


In [ ]:
# ─── CORRECCIÓN CRÍTICA ────────────────────────────────────────────────────────
# La versión original aplicaba fit() del KNN Imputer sobre el dataset completo
# ANTES del split. Esto constituye data leakage: el imputer usaba estadísticas
# de los datos de test para completar valores faltantes en el entrenamiento,
# generando una evaluación artificialmente optimista.
#
# CORRECCIÓN: el imputer se ajusta EXCLUSIVAMENTE sobre X_train.
# X_test solo recibe transform(), usando los parámetros aprendidos de train.
# ──────────────────────────────────────────────────────────────────────────────

# Paso 1: Escalado temporal para que KNN use distancias euclidianas equitativas.
#         El fit_transform se aplica SOLO a X_train.
scaler_temp = StandardScaler()
X_train_scaled_temp = scaler_temp.fit_transform(X_train)
X_test_scaled_temp  = scaler_temp.transform(X_test)   # solo transform en test

# Paso 2: KNN Imputer — fit SOLO en X_train
imputer = KNNImputer(n_neighbors=5)
X_train_imputed = imputer.fit_transform(X_train_scaled_temp)
X_test_imputed  = imputer.transform(X_test_scaled_temp)   # solo transform en test

# Paso 3: Revertir escalado temporal (el escalado definitivo viene en el siguiente paso)
X_train_imputed = scaler_temp.inverse_transform(X_train_imputed)
X_test_imputed  = scaler_temp.inverse_transform(X_test_imputed)

# Reconstruir DataFrames conservando nombres de columnas e índices
X_train = pd.DataFrame(X_train_imputed, columns=X_train.columns, index=X_train.index)
X_test  = pd.DataFrame(X_test_imputed,  columns=X_test.columns,  index=X_test.index)

print("Imputación KNN finalizada.")
print(f"Nulos restantes en X_train: {X_train.isnull().sum().sum()}")
print(f"Nulos restantes en X_test:  {X_test.isnull().sum().sum()}")


### 2.4 Escalado definitivo y SMOTE

El escalado definitivo (StandardScaler) se ajustó exclusivamente sobre `X_train` y se aplicó con `transform()` sobre `X_test`, asegurando que las estadísticas del conjunto de prueba no contaminen el entrenamiento.

SMOTE se aplicó **solo sobre el conjunto de entrenamiento** para balancear la distribución de clases durante el aprendizaje. Aplicarlo sobre el conjunto de test hubiera contaminado la evaluación con muestras sintéticas, generando métricas que no reflejan el desempeño real. El test permaneció inalterado: real y desbalanceado, tal como se encontrará el modelo en producción.


In [ ]:
# Escalado definitivo — fit SOLO en X_train
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# SMOTE — Sobremuestreo sintético de la clase minoritaria
# APLICADO SOLO SOBRE EL CONJUNTO DE ENTRENAMIENTO.
# Aplicarlo sobre test contaminaría la evaluación con muestras artificiales,
# generando métricas irreales.
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train_scaled, y_train)

print(f"Antes de SMOTE — Train: {len(y_train)} registros | Positivos: {y_train.sum()}")
print(f"Después de SMOTE — Train: {len(y_res)} registros | Positivos: {y_res.sum()}")
print(f"\n→ El conjunto de entrenamiento está ahora balanceado: {dict(pd.Series(y_res).value_counts())}")
print(f"→ El conjunto de TEST no fue modificado — sigue siendo real y desbalanceado.")


---
## 3. Entrenamiento y Comparativa de Modelos

Se entrenaron cuatro modelos de clasificación (Regresión Logística, Random Forest, XGBoost y SVM) con búsqueda de hiperparámetros mediante GridSearchCV de 5 folds. La métrica de optimización en todos los casos fue el **Recall**, en coherencia con la prioridad de minimizar falsos negativos definida en la Sección 2.

> **Corrección respecto a la versión original:** se empleaba `scoring='f1'`, métrica que no refleja la asimetría del riesgo en este problema clínico.


In [ ]:
# Definición de modelos y grilla de hiperparámetros
# Nota: DecisionTree y MLPClassifier fueron eliminados del listado porque no
# se incluyeron en la evaluación. Si se quieren agregar, deben estar en la grilla.

models = {
    'Logistic Regression': {
        'model': LogisticRegression(max_iter=1000, random_state=42),
        'params': {'C': [0.01, 0.1, 1, 10]}
    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {'n_estimators': [100, 200], 'max_depth': [10, 20, None]}
    },
    'XGBoost': {
        'model': XGBClassifier(eval_metric='logloss', random_state=42,
                               use_label_encoder=False),
        'params': {'learning_rate': [0.05, 0.1], 'n_estimators': [100, 200]}
    },
    'SVM': {
        'model': SVC(probability=True, random_state=42),
        'params': {'C': [0.1, 1, 10], 'kernel': ['rbf']}
    }
}

results = []
trained_models = {}

for name, config in models.items():
    print(f"Entrenando {name}...")
    start = time.time()

    # CORRECCIÓN: scoring='recall' (antes era 'f1')
    # Se optimiza recall para minimizar falsos negativos (ACV no detectados)
    clf = GridSearchCV(
        config['model'], config['params'],
        cv=5, scoring='recall', n_jobs=-1
    )
    clf.fit(X_res, y_res)
    best_model = clf.best_estimator_
    elapsed = time.time() - start

    y_pred  = best_model.predict(X_test_scaled)
    y_proba = best_model.predict_proba(X_test_scaled)[:, 1]

    trained_models[name] = {
        'model': best_model,
        'proba': y_proba,
        'pred': y_pred,
        'best_params': clf.best_params_
    }

    results.append({
        'Modelo': name,
        'Accuracy (%)':  round(accuracy_score(y_test, y_pred) * 100, 2),
        'Precision (%)': round(precision_score(y_test, y_pred, zero_division=0) * 100, 2),
        'Recall (%)':    round(recall_score(y_test, y_pred) * 100, 2),
        'F1-Score (%)':  round(f1_score(y_test, y_pred, zero_division=0) * 100, 2),
        'AUC-ROC':       round(roc_auc_score(y_test, y_proba), 4),
        'MCC':           round(matthews_corrcoef(y_test, y_pred), 4),
        'Tiempo (s)':    round(elapsed, 2)
    })
    r = results[-1]
    print(f"  Recall: {r['Recall (%)']:.1f}% | AUC-ROC: {r['AUC-ROC']:.4f} | "
          f"Mejores params: {clf.best_params_}\n")

df_results = pd.DataFrame(results).set_index('Modelo')
print("\n=== TABLA COMPARATIVA ===")
display(df_results)


In [ ]:
# Visualización comparativa de métricas clave
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Comparativa de Modelos — Métricas Clave', fontsize=13, fontweight='bold')

metricas = ['Recall (%)', 'AUC-ROC', 'MCC']
colores_modelo = ['#4C9BE8', '#E85D5D', '#50C878', '#FF9800']

for i, metrica in enumerate(metricas):
    vals = df_results[metrica]
    bars = axes[i].bar(df_results.index, vals, color=colores_modelo, width=0.5)
    axes[i].set_title(metrica, fontweight='bold')
    axes[i].set_ylabel(metrica)
    axes[i].tick_params(axis='x', rotation=30)
    for bar, val in zip(bars, vals):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                     f'{val:.2f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n→ Métrica de selección principal: RECALL (minimiza falsos negativos)")
print(f"→ Modelo con mayor Recall: {df_results['Recall (%)'].idxmax()} "
      f"({df_results['Recall (%)'].max():.2f}%)")


---
## 4. Evaluación del Modelo Seleccionado y Conclusiones

Se seleccionó el modelo con mayor Recall sobre el conjunto de test, que es el criterio prioritario definido para este problema. A continuación se presenta el análisis detallado de su desempeño, incluyendo la interpretación clínica de los resultados, la estimación del impacto operativo y las consideraciones para su potencial implementación en la red provincial.


In [ ]:
# Selección del modelo con mayor Recall
best_model_name = df_results['Recall (%)'].idxmax()
best = trained_models[best_model_name]
y_pred_best  = best['pred']
y_proba_best = best['proba']

print(f"Modelo seleccionado: {best_model_name}")
print(f"Recall:  {recall_score(y_test, y_pred_best)*100:.2f}%")
print(f"AUC-ROC: {roc_auc_score(y_test, y_proba_best):.4f}")
print(f"Mejores hiperparámetros: {best['best_params']}")


### 4.1 Matriz de confusión — interpretación clínica

In [ ]:
cm = confusion_matrix(y_test, y_pred_best)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Evaluación Clínica — {best_model_name}', fontsize=13, fontweight='bold')

# Matriz de confusión estándar
disp = ConfusionMatrixDisplay(cm, display_labels=['Sin ACV (0)', 'Con ACV (1)'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Matriz de Confusión')

# Interpretación clínica
categorias = [
    'Verdaderos Negativos (TN)\nDetectados sin ACV OK',
    'Falsos Positivos (FP)\nAlerta innecesaria',
    'Falsos Negativos (FN)\n⚠ ACV no detectado',
    'Verdaderos Positivos (TP)\nACV detectado OK'
]
valores  = [tn, fp, fn, tp]
colores  = ['#4CAF50', '#FF9800', '#F44336', '#2196F3']

bars = axes[1].bar(range(4), valores, color=colores, width=0.6)
axes[1].set_xticks(range(4))
axes[1].set_xticklabels(categorias, fontsize=8.5)
axes[1].set_title('Interpretación Clínica')
axes[1].set_ylabel('Cantidad de pacientes')
for bar, val in zip(bars, valores):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 str(val), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

positivos_test = int(y_test.sum())
print(f"\n═══ RESUMEN SOBRE EL CONJUNTO DE TEST ({len(y_test)} pacientes) ═══")
print(f"  Pacientes con ACV real: {positivos_test}")
print(f"  ✔ ACV detectados correctamente (TP): {tp}  ({tp/positivos_test*100:.1f}%)")
print(f"  ✘ ACV NO detectados — CRÍTICO (FN): {fn}  ({fn/positivos_test*100:.1f}%)")
print(f"  ⚠ Falsas alarmas (FP): {fp}")
print(f"  ✔ Sin ACV correctamente descartados (TN): {tn}")


### 4.2 Curvas ROC y Precision-Recall

Se evaluaron las curvas ROC y Precision-Recall para todos los modelos entrenados. La curva PR resultó más informativa en este dataset desbalanceado, ya que muestra cuántos de los pacientes alertados realmente tienen ACV (precisión) frente a cuántos ACV reales fueron detectados (recall). La curva ROC tiende a ser optimista cuando la clase negativa domina ampliamente el dataset.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colores_modelo = ['#E85D5D', '#4C9BE8', '#50C878', '#FF9800']

for (name, data), color in zip(trained_models.items(), colores_modelo):
    proba = data['proba']

    # Curva ROC
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc_val = roc_auc_score(y_test, proba)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc_val:.3f})', color=color)

    # Curva Precision-Recall
    prec, rec, _ = precision_recall_curve(y_test, proba)
    axes[1].plot(rec, prec, label=name, color=color)

# ROC
axes[0].plot([0,1],[0,1], 'k--', alpha=0.5, label='Aleatorio (AUC=0.5)')
axes[0].fill_between(fpr, tpr, alpha=0.05)
axes[0].set_xlabel('Tasa de Falsos Positivos (FPR)')
axes[0].set_ylabel('Tasa de Verdaderos Positivos (Recall)')
axes[0].set_title('Curva ROC', fontweight='bold')
axes[0].legend(loc='lower right', fontsize=9)
axes[0].grid(alpha=0.3)

# Precision-Recall
baseline = y_test.mean()
axes[1].axhline(baseline, color='k', linestyle='--', alpha=0.5,
                label=f'Baseline (prevalencia = {baseline:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Curva Precision-Recall\n(más informativa que ROC en datasets desbalanceados)',
                  fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("→ La curva PR es preferible en datasets desbalanceados porque muestra")
print("  cuántos de los alertados realmente tienen ACV (precisión), vs cuántos")
print("  ACV reales se detectaron (recall). La curva ROC puede ser optimista")
print("  cuando la clase negativa domina el dataset.")


### 4.3 Análisis de umbral de decisión (Threshold Tuning)

El umbral por defecto de 0.5 no resulta óptimo cuando la prioridad es maximizar el recall. Reducir el umbral incrementa la detección de casos positivos (menos FN) a costa de más falsas alarmas (más FP). En el contexto rural, un FN es considerablemente más costoso que un FP: el médico puede evaluar y descartar una falsa alarma con su criterio clínico, mientras que un ACV no detectado puede tener consecuencias irreversibles.

Se analizó el comportamiento de las métricas en función del umbral para identificar el punto de operación más adecuado para este sistema.


In [ ]:
# El umbral 0.5 no es óptimo cuando la prioridad es maximizar recall.
# Bajar el umbral aumenta el recall (menos FN) a costa de más FP (falsas alarmas).
# En el contexto rural, un FN es mucho más costoso que un FP.

thresholds = np.arange(0.05, 0.85, 0.05)
thresh_results = []

for t in thresholds:
    y_pred_t = (y_proba_best >= t).astype(int)
    # Evitar el caso donde solo hay una clase predicha
    if y_pred_t.sum() == 0 or (1 - y_pred_t).sum() == 0:
        continue
    cm_t = confusion_matrix(y_test, y_pred_t)
    tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
    thresh_results.append({
        'Umbral': round(t, 2),
        'Recall (%)':    round(recall_score(y_test, y_pred_t) * 100, 1),
        'Precision (%)': round(precision_score(y_test, y_pred_t, zero_division=0) * 100, 1),
        'F1 (%)':        round(f1_score(y_test, y_pred_t, zero_division=0) * 100, 1),
        'FN (ACV ignorados)': int(fn_t),
        'FP (Falsas alarmas)': int(fp_t)
    })

df_thresh = pd.DataFrame(thresh_results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Análisis de Umbral de Decisión', fontsize=13, fontweight='bold')

# Métricas vs umbral
axes[0].plot(df_thresh['Umbral'], df_thresh['Recall (%)'], 'r-o', ms=4, label='Recall')
axes[0].plot(df_thresh['Umbral'], df_thresh['Precision (%)'], 'b-o', ms=4, label='Precision')
axes[0].plot(df_thresh['Umbral'], df_thresh['F1 (%)'], 'g-o', ms=4, label='F1')
axes[0].axvline(0.5, color='gray', linestyle='--', alpha=0.7, label='Umbral default (0.5)')
axes[0].set_xlabel('Umbral de decisión')
axes[0].set_ylabel('Métrica (%)')
axes[0].set_title('Recall, Precision y F1 vs Umbral')
axes[0].legend()
axes[0].grid(alpha=0.3)

# FN y FP vs umbral
axes[1].plot(df_thresh['Umbral'], df_thresh['FN (ACV ignorados)'], 'r-o', ms=4,
             label='FN — ACV ignorados ⚠')
axes[1].plot(df_thresh['Umbral'], df_thresh['FP (Falsas alarmas)'], 'b-o', ms=4,
             label='FP — Falsas alarmas')
axes[1].axvline(0.5, color='gray', linestyle='--', alpha=0.7, label='Umbral default (0.5)')
axes[1].set_xlabel('Umbral de decisión')
axes[1].set_ylabel('Cantidad de pacientes (conjunto de test)')
axes[1].set_title('FN y FP vs Umbral')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\nTabla de umbral (selección):")
display(df_thresh.set_index('Umbral')
        .style.background_gradient(cmap='RdYlGn', subset=['Recall (%)'])
              .background_gradient(cmap='RdYlGn_r', subset=['FN (ACV ignorados)']))


### 4.4 Importancia de features y confiabilidad por grupos de pacientes

**Importancia de features:**

Se analizó la importancia relativa de cada variable en los modelos Random Forest y XGBoost. Los resultados mostraron que `edad` y `nivel_glucosa` son los predictores más relevantes, en línea con el conocimiento clínico establecido sobre factores de riesgo cardiovascular.

**Nivel de confianza del modelo y grupos de menor confiabilidad:**

El AUC-ROC y el Recall sobre el conjunto de test reflejan la capacidad discriminatoria del modelo sobre datos no vistos durante el entrenamiento. Sin embargo, se identificaron grupos de pacientes para los cuales el modelo es menos confiable:

1. **Pacientes muy jóvenes (< 20 años):** Los ACV en este rango son infrecuentes incluso clínicamente. El dataset presenta escasa representación de este grupo, por lo que el modelo dispone de poca evidencia para hacer predicciones robustas.

2. **Pacientes de edad avanzada (> 80 años):** Pueden estar sub-representados. La presencia de múltiples comorbilidades puede invalidar los patrones aprendidos por el modelo en este rango etario.

3. **Pacientes con `estado_fumador` imputado:** Las predicciones sobre pacientes cuyo estado de tabaquismo fue estimado (no reportado directamente) tienen mayor incertidumbre inherente al dato de entrada.

**Instrucción específica para el médico que usa el sistema:**
> *"Cuando atienda a un paciente menor de 25 años, mayor de 80 años, o con historia clínica incompleta sobre tabaquismo, trate la alerta del sistema como una señal débil. En estos casos, su criterio clínico tiene mayor peso que el del sistema. Ante cualquier duda, derive."*


In [ ]:
feature_names = X_train.columns.tolist()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Importancia de Features', fontsize=13, fontweight='bold')

# Random Forest
if 'Random Forest' in trained_models:
    rf = trained_models['Random Forest']['model']
    imp_rf = pd.Series(rf.feature_importances_, index=feature_names).sort_values()
    imp_rf.plot(kind='barh', ax=axes[0], color='steelblue')
    axes[0].set_title('Random Forest — Gini Importance')
    axes[0].set_xlabel('Importancia relativa')

# XGBoost
if 'XGBoost' in trained_models:
    xgb = trained_models['XGBoost']['model']
    imp_xgb = pd.Series(xgb.feature_importances_, index=feature_names).sort_values()
    imp_xgb.plot(kind='barh', ax=axes[1], color='coral')
    axes[1].set_title('XGBoost — Feature Importance (gain)')
    axes[1].set_xlabel('Importancia relativa')

plt.tight_layout()
plt.show()

# Top features
print("Top 3 features más predictivos (Random Forest):")
print(imp_rf.sort_values(ascending=False).head(3).to_string())
print("\n→ Coincide con el conocimiento clínico: edad y glucosa son")
print("  factores de riesgo cardiovascular bien documentados.")


### 4.5 Estimación del impacto clínico operativo

Se estimó el impacto práctico del modelo proyectando sus métricas sobre una muestra de 1000 pacientes atendidos, utilizando la prevalencia observada en el dataset de test.

**¿Son estas cifras éticamente aceptables para presentarlas al director médico?**

La respuesta depende del contexto de comparación. El sistema no compite contra la perfección: compite con la situación actual, donde el médico de cabecera, con 8-12 minutos por consulta y sin herramienta automatizada, también comete falsos negativos. Si el modelo detecta el 70-80% de los casos de alto riesgo que actualmente pasan desapercibidos, representa un avance significativo en salud preventiva rural.

**Condición ética irrenunciable:** Los falsos negativos deben comunicarse explícitamente al director médico. No es aceptable presentar solo el porcentaje de detección sin mencionar cuántos casos quedarían sin alerta. El director médico tiene derecho a tomar la decisión de implementación conociendo ambos lados del tradeoff.


In [ ]:
# Estimación de impacto clínico en 100 pacientes
prevalencia   = float(y_test.mean())
recall_val    = float(recall_score(y_test, y_pred_best))
precision_val = float(precision_score(y_test, y_pred_best, zero_division=0))
fp_rate       = float(fp / len(y_test[y_test == 0]))   # tasa de FP entre negativos reales

N = 1000   # por cada 1000 pacientes atendidos
positivos_esperados   = prevalencia * N
fn_esperados          = positivos_esperados * (1 - recall_val)
tp_esperados          = positivos_esperados * recall_val
negativos_esperados   = (1 - prevalencia) * N
fp_esperados          = negativos_esperados * fp_rate
derivaciones_totales  = tp_esperados + fp_esperados
derivaciones_evitadas = N - derivaciones_totales

print("═" * 55)
print(f"ESTIMACIÓN DE IMPACTO — POR CADA {N} PACIENTES ATENDIDOS")
print("═" * 55)
print(f"Prevalencia estimada de ACV: {prevalencia*100:.1f}%")
print(f"Pacientes con riesgo real:   ~{positivos_esperados:.0f}")
print(f"  ✔ Detectados (TP):         ~{tp_esperados:.0f}")
print(f"  ✘ Ignorados — CRÍTICO (FN):~{fn_esperados:.0f}")
print(f"Falsas alarmas (FP):          ~{fp_esperados:.0f}")
print(f"Derivaciones evitadas:        ~{derivaciones_evitadas:.0f}")
print(f"\nRecall del modelo: {recall_val*100:.1f}%")
print(f"Precisión del modelo: {precision_val*100:.1f}%")


---
## 5. Recomendación Final para la Red Provincial

Se analizó en conjunto la evidencia técnica y el contexto operativo para formular una recomendación sobre la implementación del sistema.

**Recomendación: SÍ, bajo las siguientes condiciones estrictas.**

**1. Rol de apoyo, nunca de reemplazo**  
El sistema emite alertas de derivación, no diagnósticos. El médico conserva plena autoridad y responsabilidad clínica. Este rol debe quedar explicitado contractualmente con la red de salud.

**2. Monitoreo continuo del desempeño**  
Cada 3-6 meses se evalúan métricas reales en producción: cuántas alertas se emitieron, cuántas derivaciones resultaron en diagnóstico confirmado. Si el Recall cae significativamente respecto al evaluado en test, el modelo se re-entrena con nuevos datos.

**3. Protocolo de falla segura**  
Ante cualquier problema técnico del sistema, el médico continúa con el protocolo clínico manual habitual. El sistema nunca debe bloquear ni condicionar la atención.

**4. Transparencia con el paciente**  
El paciente debe ser informado de la existencia de un sistema de soporte computacional y tiene derecho a conocer su funcionamiento general.

**5. Protocolos de seguridad de datos**
- Datos almacenados en servidores con acceso restringido y cifrado.
- Cumplimiento de legislación provincial/nacional de protección de datos de salud.
- Anonimización antes de cualquier uso analítico o de re-entrenamiento.

**6. Fase piloto antes del escalado**  
Se recomienda implementar el sistema en 2-3 centros piloto con seguimiento intensivo durante 6 meses. Se evaluarán el recall real, la tasa de derivaciones y el feedback de los médicos antes de escalar a la red completa.

---

**¿Es viable reemplazar completamente al médico con este sistema?**

**No.** Este sistema **no puede reemplazar el criterio médico** por las siguientes razones:

- El dataset (~5100 casos) es limitado y puede no representar la diversidad demográfica completa de la provincia.
- El modelo tiene una tasa de FN no nula — ningún sistema automático puede garantizar recall del 100%.
- El ACV puede manifestarse con síntomas agudos que no forman parte de los features del modelo (el sistema no es un monitor en tiempo real).
- La responsabilidad civil y ética del diagnóstico y la derivación recae legalmente en el médico, no en el sistema.

El sistema es una herramienta de apoyo a la decisión clínica, no un agente autónomo de diagnóstico.
